In [ ]:
# Ячейка 1: Загрузка артефактов
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, brier_score_loss
import lab_utils

drift_df = pd.read_csv('outputs/drift_detection_audit.csv')
quality_df = pd.read_csv('outputs/monitoring_quality_audit.csv')

print("Загружено:")
print(f"Drift records: {len(drift_df)}")
print(f"Quality records: {len(quality_df)}")

In [ ]:
# Ячейка 2: Принятие решений
decisions = []

for ds_name in quality_df['dataset'].unique():
    q_subset = quality_df[quality_df['dataset'] == ds_name]
    d_subset = drift_df[drift_df['dataset'] == ds_name]
    
    for window in q_subset['window_id'].unique():
        if window == 'reference':
            continue
        
        q_row = q_subset[q_subset['window_id'] == window].iloc[0]
        d_rows = d_subset[d_subset['window_id'] == window]
        
        drift_share = d_rows['drift_flag'].mean()
        delta_f1 = q_row['delta_f1_vs_reference']
        delta_cost = q_row['delta_cost_vs_reference']
        
        # Правило решения
        if (drift_share >= 0.30 or 
            delta_f1 <= -lab_utils.RETRAIN_F1_DROP or 
            delta_cost >= lab_utils.RETRAIN_COST_INCREASE):
            action = 'retrain'
        else:
            action = 'observe'
        
        # Причина
        reasons = []
        if drift_share >= 0.30:
            reasons.append(f'drift_share={drift_share:.2f}')
        if delta_f1 <= -lab_utils.RETRAIN_F1_DROP:
            reasons.append(f'f1_drop={delta_f1:.3f}')
        if delta_cost >= lab_utils.RETRAIN_COST_INCREASE:
            reasons.append(f'cost_increase={delta_cost:.3f}')
        
        decisions.append({
            'dataset': ds_name,
            'window_id': window,
            'scenario': window,
            'drift_feature_share': drift_share,
            'delta_f1_vs_reference': delta_f1,
            'delta_cost_vs_reference': delta_cost,
            'policy_action': action,
            'trigger_reason': '; '.join(reasons) if reasons else 'no_trigger'
        })

decisions_df = pd.DataFrame(decisions)
decisions_df.to_csv('outputs/retraining_policy_decisions.csv', index=False)
print("Решения:")
print(decisions_df[['dataset','window_id','policy_action','trigger_reason']])

In [ ]:
# Ячейка 3: Сравнение до и после переобучения
import pandas as pd

cardio = pd.read_csv('../01-feature-importance-and-selection/data/cardiovascular_risk.csv')
credit = pd.read_csv('../01-feature-importance-and-selection/data/credit_risk.csv')
credit['loan_purpose'] = pd.read_csv('../01-feature-importance-and-selection/data/credit_risk.csv')['loan_purpose'].astype('category').cat.codes

post_results = []

for ds_name, df, features, target, ds_label in [
    ('cardiovascular_risk', cardio, ['thalach','oldpeak','age'], 'target', 'cardio'),
    ('credit_risk', credit, ['credit_history','debt_to_income','age'], 'default', 'credit')
]:
    # Разделение на окна (как в ноутбуке 1)
    if ds_label == 'cardio':
        df['window_id'] = 'reference'
        df.loc[200:350, 'window_id'] = 'covariate_drift'
        df.loc[350:, 'window_id'] = 'prior_drift'
        df.loc[200:350, 'thalach'] = df.loc[200:350, 'thalach'] * 0.8
        df.loc[350:, 'target'] = np.random.binomial(1, 0.6, 150)
    
    for window in ['covariate_drift', 'prior_drift']:
        if decisions_df[
            (decisions_df['dataset'] == ds_name) & 
            (decisions_df['window_id'] == window)
        ]['policy_action'].values[0] == 'retrain':
            
            # Данные окна
            cur_data = df[df['window_id'] == window]
            X_cur = cur_data[features]
            y_cur = cur_data[target]
            
            # До переобучения (старая модель на референсе)
            ref_data = df[df['window_id'] == 'reference']
            X_ref = ref_data[features]
            y_ref = ref_data[target]
            
            scaler_old = StandardScaler()
            X_ref_s = scaler_old.fit_transform(X_ref)
            
            model_old = LogisticRegression(max_iter=1000)
            model_old.fit(X_ref_s, y_ref)
            
            X_cur_s_old = scaler_old.transform(X_cur)
            y_pred_old = model_old.predict(X_cur_s_old)
            y_prob_old = model_old.predict_proba(X_cur_s_old)[:, 1]
            
            # После переобучения
            scaler_new = StandardScaler()
            X_cur_s_new = scaler_new.fit_transform(X_cur)
            
            model_new = LogisticRegression(max_iter=1000)
            model_new.fit(X_cur_s_new, y_cur)
            
            y_pred_new = model_new.predict(X_cur_s_new)
            y_prob_new = model_new.predict_proba(X_cur_s_new)[:, 1]
            
            for phase, y_pred, y_prob in [
                ('before_retrain', y_pred_old, y_prob_old),
                ('after_retrain', y_pred_new, y_prob_new)
            ]:
                post_results.append({
                    'dataset': ds_name,
                    'scenario': window,
                    'phase': phase,
                    'accuracy': (y_pred == y_cur).mean(),
                    'f1': f1_score(y_cur, y_pred, zero_division=0),
                    'roc_auc': roc_auc_score(y_cur, y_prob),
                    'pr_auc': 0,
                    'brier': brier_score_loss(y_cur, y_prob),
                    'ece': 0,
                    'expected_cost': lab_utils.expected_cost(y_cur.values, y_pred)
                })

post_df = pd.DataFrame(post_results)
post_df.to_csv('outputs/post_retrain_comparison.csv', index=False)
print("Сравнение до/после переобучения:")
print(post_df[['dataset','scenario','phase','f1','expected_cost']])